# Caso Practico: Creatividad Digital

**Version instructor (referencia).**


In [1]:
import pandas as pd
from pathlib import Path

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    plt = None
    print('Aviso: matplotlib no esta instalado. Se omitiran graficas.')

DATA_PATH = Path('datos/campanas_creatividad_digital.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('Casos_Practicos/Creatividad_Digital/datos/campanas_creatividad_digital.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['fecha'])
df.head()


,id_campana,fecha,cliente,plataforma,formato,duracion_seg,objetivo,pais,presupuesto_usd,costo_usd,impresiones,alcance,clicks,reproducciones_3s,reproducciones_100,conversiones,ingresos_usd,calificacion_creativa
0,CAMP-0001,2025-01-02,LumaAds,YouTube,Reel,30,Traffic,MX,2449.27,2356.84,319655,212076,4073,171648,26034,82,626.92,79
1,CAMP-0002,2025-01-03,FrameHouse,Facebook,Reel,25,Traffic,AR,1153.08,836.40,185919,103138,3718,88732,22749,93,678.65,93
2,CAMP-0003,2025-01-04,PixelForge,Facebook,Video Largo,60,Ventas,PE,398.93,291.12,50288,27514,488,22772,9557,24,1082.33,63
3,CAMP-0004,2025-01-05,FrameHouse,YouTube,Video Largo,90,Awareness,AR,1752.35,1304.59,279213,210519,2491,177098,29152,19,64.42,69
4,CAMP-0005,2025-01-06,NeoStudio,Instagram,Video Largo,45,Ventas,CL,703.80,640.98,140856,109020,1878,68165,39455,137,5787.29,73


In [2]:
print('Shape:', df.shape)
print()
print('Tipos:')
print(df.dtypes)
print()
print('Nulos:')
print(df.isna().sum())


Shape: (180, 18)

Tipos:
id_campana                       object
fecha                    datetime64[ns]
cliente                          object
plataforma                       object
formato                          object
duracion_seg                      int64
objetivo                         object
pais                             object
presupuesto_usd                 float64
costo_usd                       float64
impresiones                       int64
alcance                           int64
clicks                            int64
reproducciones_3s                 int64
reproducciones_100                int64
conversiones                      int64
ingresos_usd                    float64
calificacion_creativa             int64
dtype: object

Nulos:
id_campana               0
fecha                    0
cliente                  0
plataforma               0
formato                  0
duracion_seg             0
objetivo                 0
pais                     0
presupuesto_usd  

In [3]:
df.clicks

0       4073
1       3718
2        488
3       2491
4       1878
       ...  
175     2364
176     5831
177     5467
178     8112
179    10367
Name: clicks, Length: 180, dtype: int64

In [ ]:
df['ctr'] = df['clicks'] / df['impresiones']


In [ ]:
df['cvr'] = df['conversiones'] / df['clicks'].replace(0, 1)
df['cpc'] = df['costo_usd'] / df['clicks'].replace(0, 1)
df['cpm'] = (df['costo_usd'] / df['impresiones'].replace(0, 1)) * 1000
df['roi'] = (df['ingresos_usd'] - df['costo_usd']) / df['costo_usd'].replace(0, 1)

for col in ['ctr', 'cvr', 'cpc', 'cpm', 'roi']:
    df[col] = df[col].fillna(0)

df[['ctr', 'cvr', 'cpc', 'cpm', 'roi']].describe()


In [ ]:
q1 = df[(df['roi'] < 0) & (df['costo_usd'] > df['costo_usd'].median())]
q2 = df[df['plataforma'].isin(['TikTok', 'Instagram'])].sort_values('roi', ascending=False).head(10)
q3 = df.groupby('formato', as_index=False)['ctr'].mean().sort_values('ctr', ascending=False)

print('ROI negativo + costo alto:', q1.shape[0])
q2[['id_campana', 'plataforma', 'formato', 'roi']].head()


In [ ]:
resumen_global = df[['costo_usd', 'ingresos_usd', 'ctr', 'cvr', 'cpc', 'cpm', 'roi']].describe().T
por_plataforma = df.groupby('plataforma')[['ctr', 'cvr', 'cpc', 'cpm', 'roi']].agg(['mean', 'median', 'std'])
por_formato = df.groupby('formato')[['ctr', 'roi']].agg(['mean', 'median', 'std'])

resumen_global


In [ ]:
q1_ctr = df['ctr'].quantile(0.25)
q3_ctr = df['ctr'].quantile(0.75)
iqr = q3_ctr - q1_ctr
lim_inf = q1_ctr - 1.5 * iqr
lim_sup = q3_ctr + 1.5 * iqr

outliers_ctr = df[(df['ctr'] < lim_inf) | (df['ctr'] > lim_sup)]
print('Outliers CTR:', len(outliers_ctr))
outliers_ctr[['id_campana', 'plataforma', 'formato', 'ctr', 'calificacion_creativa']].head()


In [ ]:
mensual = df.copy()
mensual['mes'] = mensual['fecha'].dt.to_period('M').astype(str)
trend = mensual.groupby('mes', as_index=False)[['costo_usd', 'ingresos_usd']].sum()

if HAS_MPL:
    plt.figure(figsize=(10, 4))
    plt.plot(trend['mes'], trend['costo_usd'], marker='o', label='Costo')
    plt.plot(trend['mes'], trend['ingresos_usd'], marker='o', label='Ingresos')
    plt.xticks(rotation=45)
    plt.title('Costo vs Ingresos por Mes')
    plt.legend()
    plt.tight_layout()
    plt.show()

    roi_plat = df.groupby('plataforma', as_index=False)['roi'].mean().sort_values('roi', ascending=False)
    plt.figure(figsize=(8, 4))
    plt.bar(roi_plat['plataforma'], roi_plat['roi'])
    plt.title('ROI Promedio por Plataforma')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 5))
    for plataforma, sub in df.groupby('plataforma'):
        plt.scatter(sub['calificacion_creativa'], sub['ctr'], label=plataforma, alpha=0.6)
    plt.title('Creatividad vs CTR')
    plt.xlabel('Calificacion Creativa')
    plt.ylabel('CTR')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Graficas omitidas por falta de matplotlib.')
    print('Resumen mensual costo/ingresos:')
    print(trend.head())


In [ ]:
top_roi = df.sort_values('roi', ascending=False).head(5)
peor_roi = df.sort_values('roi', ascending=True).head(5)
mejor_formato = df.groupby('formato')['ctr'].mean().idxmax()
mejor_plataforma_roi = df.groupby('plataforma')['roi'].mean().idxmax()

top_texto = top_roi[['id_campana', 'plataforma', 'formato', 'roi']].to_string(index=False)
peor_texto = peor_roi[['id_campana', 'plataforma', 'formato', 'roi']].to_string(index=False)

reporte = f"""# Reporte de Analisis - Caso Creatividad Digital

## Resumen Ejecutivo
- Total de campanas analizadas: {len(df)}
- Costo total: ${df['costo_usd'].sum():,.2f}
- Ingresos totales: ${df['ingresos_usd'].sum():,.2f}
- ROI promedio: {df['roi'].mean():.2f}

## Hallazgos Clave
1. Mejor formato por CTR: **{mejor_formato}**.
2. Mejor plataforma por ROI promedio: **{mejor_plataforma_roi}**.
3. Se detectaron {len(outliers_ctr)} outliers en CTR (metodo IQR).

## Top 5 Campanas por ROI
```text
{top_texto}
```

## Bottom 5 Campanas por ROI
```text
{peor_texto}
```

## Recomendaciones
- Reasignar presupuesto hacia plataformas y formatos con ROI superior a la mediana.
- Revisar creatividades de campanas con ROI negativo y costo alto.
- Implementar monitoreo semanal de CTR y CVR para ajuste temprano.
"""

with open('reporte_caso_creatividad.md', 'w', encoding='utf-8') as f:
    f.write(reporte)

print('Reporte generado: reporte_caso_creatividad.md')
